# Évaluation : détection de sons anormaux de machines (DCASE 2020 Task 2)

Ce notebook charge un modèle entraîné par `python train.py --config configs/ae.yaml` sur les sons normaux
d'un type de machine (par défaut la pompe, `data/pump/` créé par `python prepare_data.py`), et mesure s'il repère
les sons anormaux du jeu de test : fuite, contamination, colmatage…

**Protocole du papier** (Koizumi et al., 2020, *Description and Discussion on DCASE2020 Challenge Task2*) :
- chaque son de 10 s est transformé en log-mel (128 bandes, 313 trames), puis en vecteurs de 5 trames consécutives
  (640 valeurs) ;
- le **score d'anomalie** d'un son est l'erreur de reconstruction moyenne sur tous ses vecteurs ;
- les métriques sont calculées **par machine (ID)** : AUC, et pAUC (AUC partielle pour un taux de faux positifs
  ≤ 0.1, la zone utile en pratique), puis moyennées.

En complément du papier : un seuil de décision, quantile `QUANTILE` des scores de la validation (sons normaux
jamais vus pendant l'apprentissage des poids).

La section **Tester ses propres sons** permet d'écouter et d'essayer n'importe quel son.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio, display
from scipy.io import wavfile
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from data import N_MELS, SAMPLE_RATE, load_wav, logmel, to_vectors  # noqa: E402
from models import build_autoencoder  # noqa: E402

# ----- Paramètres -----
CHECKPOINT = None   # ex. "outputs/ae/pump/checkpoints/model.pt" ; None = le plus récent
QUANTILE = 0.95     # seuil = quantile des scores de validation -> ~5 % de fausses alertes
MAX_FPR = 0.1       # pAUC du papier
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ImportError: cannot import name 'N_MELS' from 'data' (c:\Users\samue\OneDrive\Documents\Projets\detection_anomalies\src\data.py)

## Chargement du modèle

In [ ]:
if CHECKPOINT is None:
    CHECKPOINT = max((ROOT / "outputs").glob("*/*/checkpoints/model.pt"), key=lambda p: p.stat().st_mtime)
ckpt = torch.load(ROOT / CHECKPOINT, map_location="cpu", weights_only=True)
cfg = ckpt["config"]

model = build_autoencoder(cfg["model"])
model.load_state_dict(ckpt["model_state"])
model = model.to(DEVICE).eval()

N_FRAMES = cfg["data"].get("n_frames", 5)
DATA = ROOT / cfg["data"]["root"]
MACHINE = DATA.name
RAW = ROOT / "data" / "dcase2020" / MACHINE  # sons WAV d'origine, pour les écouter
print(f"Checkpoint : {Path(CHECKPOINT).relative_to(ROOT) if Path(CHECKPOINT).is_absolute() else CHECKPOINT}")
print(f"Modèle : {cfg['model']['name']} | machine : {MACHINE} | {ckpt['epochs']} epochs")

In [ ]:
h = ckpt["history"]
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(h["train"]) + 1), h["train"], label="train")
plt.plot(range(1, len(h["val"]) + 1), h["val"], label="validation")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.title("Courbes d'apprentissage")
plt.legend()
plt.show()

## Données

In [ ]:
def load_folder(folder):
    """Log-mel (.npy) d'un dossier : liste de tableaux (128, trames) et noms des sons."""
    paths = sorted(folder.glob("*.npy"))
    return [np.load(p) for p in paths], [p.stem for p in paths]


mels_val, _ = load_folder(DATA / "val" / "normal")
mels_normal, names_normal = load_folder(DATA / "test" / "normal")
mels_anomaly, names_anomaly = load_folder(DATA / "test" / "anomalie")

# Test : une ligne par son. Le nom contient l'ID de la machine : normal_id_00_00000000 -> id_00
mels_test = mels_normal + mels_anomaly
test = pd.DataFrame({"name": names_normal + names_anomaly,
                     "anomaly": [False] * len(mels_normal) + [True] * len(mels_anomaly)})
test["machine_id"] = test["name"].str.extract(r"(id_\d+)")[0]

print(f"Validation : {len(mels_val)} sons normaux")
print(f"Test : {len(test)} sons dont {test.anomaly.sum()} anormaux ({test.anomaly.mean():.1%})")
test.groupby(["machine_id", "anomaly"]).size().unstack()

## Scores d'anomalie

In [ ]:
@torch.no_grad()
def score(mel):
    """Log-mel d'un son -> score (plus grand = plus suspect), moyennés sur tous ses vecteurs de 5 trames."""
    x = to_vectors(mel, N_FRAMES).to(DEVICE)
    rec = model(x)
    return {"mse": ((rec - x) ** 2).mean().item()}  # score du papier


scores_val = pd.DataFrame([score(m) for m in mels_val])
test = test.join(pd.DataFrame([score(m) for m in mels_test]))
SCORES = list(scores_val.columns)
threshold = scores_val.quantile(QUANTILE)
test.head()

## Métriques du papier : AUC et pAUC par machine

`roc_auc_score(..., max_fpr=0.1)` est la pAUC calculée par le code de la baseline (version standardisée de
scikit-learn). La dernière colonne rappelle les résultats de la baseline officielle (moyenne de 10 entraînements),
pour la pompe.

In [ ]:
# Baseline officielle, pompe (Koizumi et al., 2020) : (AUC, pAUC) en %
BASELINE = {"pump": {"id_00": (67.15, 56.74), "id_02": (61.53, 58.10), "id_04": (88.33, 67.10), "id_06": (74.55, 58.02)}}

rows = []
for machine_id, g in test.groupby("machine_id"):
    row = {"machine_id": machine_id}
    for name in SCORES:
        row[f"AUC {name}"] = 100 * roc_auc_score(g.anomaly, g[name])
        row[f"pAUC {name}"] = 100 * roc_auc_score(g.anomaly, g[name], max_fpr=MAX_FPR)
    if machine_id in BASELINE.get(MACHINE, {}):
        row["AUC baseline"], row["pAUC baseline"] = BASELINE[MACHINE][machine_id]
    rows.append(row)
results = pd.DataFrame(rows).set_index("machine_id")
results.loc["moyenne"] = results.mean()
results.style.format("{:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for machine_id, g in test.groupby("machine_id"):
    fpr, tpr, _ = roc_curve(g.anomaly, g["mse"])
    axes[0].plot(fpr, tpr, label=f"{machine_id} (AUC {results.loc[machine_id, 'AUC mse']:.1f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].axvspan(0, MAX_FPR, color="grey", alpha=0.15, label="zone de la pAUC")
axes[0].set(xlabel="taux de faux positifs", ylabel="taux de vrais positifs", title="Courbes ROC (score mse)")
axes[0].legend(fontsize=8)

bins = np.histogram_bin_edges(test["mse"], 60)
axes[1].hist(test.loc[~test.anomaly, "mse"], bins, alpha=0.6, density=True, label="normaux")
axes[1].hist(test.loc[test.anomaly, "mse"], bins, alpha=0.6, density=True, label="anormaux")
axes[1].axvline(threshold["mse"], color="k", ls="--", label="seuil")
axes[1].set(title="Distribution du score mse", xlabel="mse")
axes[1].legend()
plt.tight_layout()
plt.show()

### Au seuil de validation (complément au papier)

Un seuil unique pour toutes les machines : quantile `QUANTILE` des scores des sons normaux de validation.

In [ ]:
def at_threshold(g, name):
    pred = g[name] > threshold[name]
    tp, fp = (pred & g.anomaly).sum(), (pred & ~g.anomaly).sum()
    return pd.Series({"rappel": tp / g.anomaly.sum(), "faux positifs": fp / (~g.anomaly).sum(),
                      "précision": tp / max(pred.sum(), 1)})


test.groupby("machine_id").apply(lambda g: at_threshold(g, "mse")).style.format("{:.2f}")

## Examiner des sons

`show(mels, titles, wavs)` affiche, pour chaque son, le log-mel, sa reconstruction et l'erreur de reconstruction
(chaque colonne est la première trame d'un vecteur), avec le score et la décision au seuil. Si `wavs` est donné,
un lecteur audio permet d'écouter le son.

In [ ]:
SCORE = "mse"


@torch.no_grad()
def show(mels, titles=None, wavs=None, score_name=None):
    score_name = score_name or SCORE
    for i, mel in enumerate(mels):
        x = to_vectors(mel, N_FRAMES).to(DEVICE)
        rec = model(x).cpu()[:, :N_MELS].T.numpy()  # (128, vecteurs)
        orig = mel[:, : rec.shape[1]]
        s = score(mel)[score_name]
        suspect = s > threshold[score_name]
        fig, axes = plt.subplots(1, 3, figsize=(14, 2.6))
        vmin, vmax = orig.min(), orig.max()
        axes[0].imshow(orig, origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
        axes[1].imshow(rec, origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
        im = axes[2].imshow((orig - rec) ** 2, origin="lower", aspect="auto", cmap="hot")
        plt.colorbar(im, ax=axes[2])
        for ax, name in zip(axes, ["log-mel", "reconstruction", "erreur²"]):
            ax.set(title=name, xlabel="trame", ylabel="bande mel")
        title = titles[i] if titles is not None else f"son {i}"
        fig.suptitle(f"{title} | {score_name} = {s:.2f} -> {'ANORMAL' if suspect else 'normal'}",
                     color="red" if suspect else "black")
        plt.tight_layout()
        plt.show()
        if wavs is not None and wavs[i] is not None:
            display(Audio(str(wavs[i])))


def show_test(index):
    """Sons du test, par numéro de ligne de `test`."""
    rows = test.loc[list(index)]
    show([mels_test[i] for i in rows.index],
         [f"{r.name} ({'anormal' if r.anomaly else 'normal'})" for r in rows.itertuples()],
         [RAW / "test" / f"{r.name}.wav" if (RAW / "test" / f"{r.name}.wav").is_file() else None
          for r in rows.itertuples()])


# Un son normal et un son anormal de chaque machine
show_test(test.groupby(["machine_id", "anomaly"]).head(1).index[:4])

In [ ]:
# Sons les plus suspects, anomalies manquées (score le plus bas) et fausses alertes (normaux au score le plus haut)
for title, index in [("Les plus suspects", test[SCORE].nlargest(3).index),
                     ("Anomalies manquées", test[test.anomaly][SCORE].nsmallest(3).index),
                     ("Fausses alertes", test[~test.anomaly][SCORE].nlargest(3).index)]:
    print(title)
    show_test(index)

## Tester ses propres sons

N'importe quel son peut être passé à `show` après `logmel` : sons du test choisis à la main, sons modifiés
(bruit, sifflement, volume…), ou un fichier WAV à soi (ré-échantillonné à 16 kHz).

In [ ]:
# 1. Des sons choisis par leur nom dans le test
names = ["normal_id_00_00000000", "anomaly_id_00_00000000"]
show_test(test.index[test.name.isin(names)])

In [ ]:
# 2. Un son normal modifié : à partir de quand devient-il suspect ?
wav = RAW / "test" / f"{names_normal[0]}.wav"
signal = load_wav(wav)
t = np.arange(len(signal)) / SAMPLE_RATE
rng = np.random.default_rng(0)
variants = {
    "original": signal,
    "bruit blanc": signal + 0.01 * rng.normal(size=len(signal)),
    "sifflement 3 kHz": signal + 0.01 * np.sin(2 * np.pi * 3000 * t),
    "volume x 2": 2 * signal,
    "chocs": signal + 0.3 * (rng.random(len(signal)) < 0.0005) * rng.normal(size=len(signal)),
    "à l'envers": signal[::-1].copy(),
}
for name, s in variants.items():
    show([logmel(s.astype(np.float32))], [name])
    display(Audio(s, rate=SAMPLE_RATE))

In [ ]:
# 3. Un fichier WAV à soi
def load_sound(path):
    """WAV quelconque -> signal mono float32 à 16 kHz."""
    rate, s = wavfile.read(path)
    s = s / 32768.0 if s.dtype == np.int16 else s.astype(np.float32)
    s = s[:, 0] if s.ndim > 1 else s
    s = torch.as_tensor(s, dtype=torch.float32)
    return torchaudio.functional.resample(s, rate, SAMPLE_RATE).numpy() if rate != SAMPLE_RATE else s.numpy()


# s = load_sound("mon_son.wav")
# show([logmel(s)], ["mon son"]); display(Audio(s, rate=SAMPLE_RATE))

## Espace latent

Code moyen de chaque son du test (moyenne des codes de ses vecteurs), projeté en 2D par ACP.

In [ ]:
with torch.no_grad():
    codes = np.stack([model.encode(to_vectors(m, N_FRAMES).to(DEVICE)).mean(0).cpu().numpy() for m in mels_test])
z2 = PCA(n_components=2).fit_transform(codes)

plt.figure(figsize=(6, 5))
for machine_id, marker in zip(sorted(test.machine_id.unique()), "osD^"):
    for anomaly, color in [(False, "tab:blue"), (True, "tab:red")]:
        m = ((test.machine_id == machine_id) & (test.anomaly == anomaly)).values
        plt.scatter(*z2[m].T, s=10, marker=marker, c=color, alpha=0.6,
                    label=f"{machine_id} {'anormal' if anomaly else 'normal'}")
plt.title("Codes latents moyens du test (ACP 2D)")
plt.legend(fontsize=7, ncol=2)
plt.show()